killing app2: 
pkill -f "/home/manojcloudvm/infra-lab/app1/app.py" ----> kill did not work

more carefuly, pkill -f "/home/manojcloudvm/infra-lab/app1/venv/bin/python app.py"

other alts, pkill -f "app1"

curl -ik https://136.114.99.52/
curl -i "http://127.0.0.1/"
curl -i "http://127.0.0.1/proxy-dep"



In [ ]:
Errors:

manojcloudvm@instance-20260404-024439:~$ curl -i "http://127.0.0.1/"
HTTP/1.1 502 Bad Gateway
Server: nginx/1.22.1
Date: Tue, 07 Apr 2026 02:07:30 GMT
Content-Type: text/html
Content-Length: 157
Connection: keep-alive

<html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.22.1</center>
</body>
</html>

manojcloudvm@instance-20260404-024439:~$ curl -i "http://127.0.0.1:5001/proxy-dep"
HTTP/1.1 404 NOT FOUND
Server: Werkzeug/3.1.8 Python/3.11.2
Date: Tue, 07 Apr 2026 02:21:20 GMT
Content-Type: text/html; charset=utf-8
Content-Length: 207
Connection: close

<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



> sudo nginx -T | less


# configuration file /etc/nginx/sites-enabled/app1:
server {
    listen 80;
    listen [::]:80;
    server_name _;

    access_log /var/log/nginx/app1_access.log;
    error_log /var/log/nginx/app1_error.log;

    location / {
        proxy_pass http://127.0.0.1:5000;
        proxy_http_version 1.1;

        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }
}

server {
    listen 443 ssl;
    listen [::]:443 ssl;
    server_name _;

    ssl_certificate /etc/nginx/ssl/app1.crt;
    ssl_certificate_key /etc/nginx/ssl/app1.key;

    access_log /var/log/nginx/app1_ssl_access.log;
    error_log /var/log/nginx/app1_ssl_error.log;

    location / {
        proxy_pass http://127.0.0.1:5000;
        proxy_http_version 1.1;

        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }
}


> sudo tail -n 50 /home/manojcloudvm/infra-lab/app1/app1.log ---> no use right now because....

If app1 is not listening on 5000, then Nginx tries to connect to app1 and fails before app1 can receive or log anything.

> instead ---> cat /var/log/nginx/app1_error.log

manojcloudvm@instance-20260404-024439:~/infra-lab/app1$ cat /var/log/nginx/app1_error.log
2026/04/07 02:07:30 [error] 94791#94791: *4547 connect() failed (111: Connection refused) while connecting to upstream, client: 127.0.0.1, server: _, request: "GET / HTTP/1.1", upstream: "http://127.0.0.1:5000/", host: "127.0.0.1"
2026/04/07 02:07:37 [error] 94791#94791: *4549 connect() failed (111: Connection refused) while connecting to upstream, client: 127.0.0.1, server: _, request: "GET /proxy-dep HTTP/1.1", upstream: "http://127.0.0.1:5000/proxy-dep", host: "127.0.0.1"
2026/04/07 02:11:24 [error] 94791#94791: *4555 connect() failed (111: Connection refused) while connecting to upstream, client: 35.233.40.58, server: _, request: "GET / HTTP/1.1", upstream: "http://127.0.0.1:5000/", host: "136.114.99.52"
2026/04/07 02:14:37 [error] 94791#94791: *4565 connect() failed (111: Connection refused) while connecting to upstream, client: 198.235.24.9, server: _, request: "GET / HTTP/1.1", upstream: "http://127.0.0.1:5000/", host: "136.114.99.52:80"
2026/04/07 02:18:29 [error] 94791#94791: *4567 connect() failed (111: Connection refused) while connecting to upstream, client: 206.189.126.20, server: _, request: "GET / HTTP/1.1", upstream: "http://127.0.0.1:5000/", host: "136.114.99.52"
2026/04/07 02:18:29 [error] 94791#94791: *4569 connect() failed (111: Connection refused) while connecting to upstream, client: 206.189.126.20, server: _, request: "GET /favicon.ico HTTP/1.1", upstream: "http://127.0.0.1:5000/favicon.ico", host: "136.114.99.52", referrer: "http://136.114.99.52/"
2026/04/07 02:34:19 [error] 94791#94791: *4577 connect() failed (111: Connection refused) while connecting to upstream, client: 147.185.133.188, server: _, request: "GET / HTTP/1.1", upstream: "http://127.0.0.1:5000/", host: "136.114.99.52:80"

clearly confirms:

upstream is the issue.

App 1 restart: 
nohup ~/infra-lab/app1/venv/bin/python ~/infra-lab/app1/app.py > ~/infra-lab/app1/app1.log 2>&1 &